In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# 1. Import necessary libraries
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import glob
import scipy.io as sio

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

from skimage.transform import resize
from skimage import util
from sklearn.metrics import jaccard_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")


In [ ]:
# ============================================================
# 2. Define U-Net model
# ============================================================
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, init_features=32):
        super(UNet, self).__init__()

        features = init_features
        self.encoder1 = self._block(in_channels, features, name="enc1")
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder2 = self._block(features, features * 2, name="enc2")
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder3 = self._block(features * 2, features * 4, name="enc3")
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder4 = self._block(features * 4, features * 8, name="enc4")
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = self._block(features * 8, features * 16, name="bottleneck")

        self.upconv4 = nn.ConvTranspose2d(
            features * 16, features * 8, kernel_size=2, stride=2
        )
        self.decoder4 = self._block((features * 8) * 2, features * 8, name="dec4")
        self.upconv3 = nn.ConvTranspose2d(
            features * 8, features * 4, kernel_size=2, stride=2
        )
        self.decoder3 = self._block((features * 4) * 2, features * 4, name="dec3")
        self.upconv2 = nn.ConvTranspose2d(
            features * 4, features * 2, kernel_size=2, stride=2
        )
        self.decoder2 = self._block((features * 2) * 2, features * 2, name="dec2")
        self.upconv1 = nn.ConvTranspose2d(
            features * 2, features, kernel_size=2, stride=2
        )
        self.decoder1 = self._block(features * 2, features, name="dec1")

        self.conv = nn.Conv2d(
            in_channels=features, out_channels=out_channels, kernel_size=1
        )
        self.sigmoid = nn.Sigmoid()

    def _block(self, in_channels, features, name):
        return nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=features,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(num_features=features),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                in_channels=features,
                out_channels=features,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(num_features=features),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool1(enc1))
        enc3 = self.encoder3(self.pool2(enc2))
        enc4 = self.encoder4(self.pool3(enc3))

        bottleneck = self.bottleneck(self.pool4(enc4))

        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.decoder4(dec4)
        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.decoder3(dec3)
        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2(dec2)
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1(dec1)
        
        logits = self.conv(dec1)
        return self.sigmoid(logits) 

# ============================================================
# 3. Prepare Dataset and DataLoader
# ============================================================
class OpticDiscDataset(Dataset):
    def __init__(self, image_paths, mat_paths, target_size=(384, 384)):
        self.image_paths = image_paths
        self.mat_paths = mat_paths
        self.target_size = target_size

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        image = np.array(image)
        image = resize(image, self.target_size, anti_aliasing=True)
        image = torch.FloatTensor(image).permute(2, 0, 1)
        mat_path = self.mat_paths[idx]
        mat_data = sio.loadmat(mat_path)
        mask = mat_data['mask']
        disc_mask = (mask > 0).astype(np.float32)
        disc_mask = resize(disc_mask, self.target_size, anti_aliasing=False, order=0) 
        disc_mask = (disc_mask > 0.5).astype(np.float32) 
        disc_mask = torch.FloatTensor(disc_mask).unsqueeze(0)
        return image, disc_mask

# ============================================================
# 4. Training and Validation
# ============================================================
image_dir = "/kaggle/input/glaucoma-detection/ORIGA/ORIGA/Images/"
mat_dir = "/kaggle/input/glaucoma-detection/ORIGA/ORIGA/Semi-automatic-annotations/"

image_paths = sorted(glob.glob(os.path.join(image_dir, "*.jpg")))
mat_paths = sorted(glob.glob(os.path.join(mat_dir, "*.mat")))

print(f"{len(image_paths)} images found in {image_dir}")
print(f"{len(mat_paths)} mat files found in {mat_dir}")

matched_image_paths = []
matched_mat_paths = []
for img_path in image_paths:
    img_stem = Path(img_path).stem
    for mat_path in mat_paths:
        if Path(mat_path).stem == img_stem:
            matched_image_paths.append(img_path)
            matched_mat_paths.append(mat_path)
            break

total_size = len(matched_image_paths)
val_size = int(0.2 * total_size)
train_size = total_size - val_size

train_paths, val_paths = train_test_split(
    list(zip(matched_image_paths, matched_mat_paths)),
    test_size=0.2,
    random_state=42,
    shuffle=True
)
train_image_paths, train_mat_paths = zip(*train_paths)
val_image_paths, val_mat_paths = zip(*val_paths)

print(f"train data: {len(train_image_paths)}")
print(f"validation data: {len(val_image_paths)}")

train_dataset = OpticDiscDataset(train_image_paths, train_mat_paths, target_size=(384, 384))
val_dataset = OpticDiscDataset(val_image_paths, val_mat_paths, target_size=(384, 384))

batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# ============================================================
# 5. Initialize model, loss function, optimizer, scheduler
# ============================================================
model = UNet(in_channels=3, out_channels=1).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True)

# ============================================================     
# 6. training and validation loops
# ============================================================
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    loop = tqdm(loader, desc="Training")
    for images, masks in loop:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    epoch_loss = running_loss / len(loader)
    return epoch_loss

def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        loop = tqdm(loader, desc="Validating")
        for images, masks in loop:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            running_loss += loss.item()
            preds = (outputs.cpu().numpy() > 0.5).astype(np.uint8)
            targets = masks.cpu().numpy().astype(np.uint8)
            batch_size = preds.shape[0]
            h, w = preds.shape[2], preds.shape[3]
            batch_flat_size = batch_size * h * w
            all_preds.extend(preds.reshape(batch_flat_size))
            all_targets.extend(targets.reshape(batch_flat_size))
            loop.set_postfix(loss=loss.item())
    epoch_loss = running_loss / len(loader)
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    if np.sum(all_targets) == 0 and np.sum(all_preds) == 0:
        iou = 1.0  
        precision = 1.0
        recall = 1.0
        f1 = 1.0
    else:
        iou = jaccard_score(all_targets, all_preds, average='binary', zero_division=0)
        precision = precision_score(all_targets, all_preds, average='binary', zero_division=0)
        recall = recall_score(all_targets, all_preds, average='binary', zero_division=0)
        f1 = f1_score(all_targets, all_preds, average='binary', zero_division=0)

    return epoch_loss, iou, precision, recall, f1

# ============================================================
# 7. Main training loop with model saving
# ============================================================
train_losses = []
val_losses = []
val_ious = []
val_precisions = []
val_recalls = []
val_f1s = []

num_epochs = 50 
best_val_iou = 0.0
patience = 10
patience_counter = 0

model_weights_save_path = "best_unet_optic_disc_weights.pth"  
training_metadata_save_path = "best_unet_optic_disc_metadata.npy" 
history_save_path = "training_history.npy"

print("starting training...")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 10)
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_iou, val_precision, val_recall, val_f1 = validate_epoch(model, val_loader, criterion, device)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_ious.append(val_iou)
    val_precisions.append(val_precision)
    val_recalls.append(val_recall)
    val_f1s.append(val_f1)
    print(f"Train Loss: {train_loss:.6f}")
    print(f"Val Loss: {val_loss:.6f}")
    print(f"Val IoU: {val_iou:.4f}")
    print(f"Val Precision: {val_precision:.4f}")
    print(f"Val Recall: {val_recall:.4f}")
    print(f"Val F1-Score: {val_f1:.4f}")
    scheduler.step(val_iou)
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        patience_counter = 0
        torch.save(model.state_dict(), model_weights_save_path)
        metadata = {
            'epoch': epoch,
            'val_iou': val_iou,
            'val_loss': val_loss,
            'val_precision': val_precision,
            'val_recall': val_recall,
            'val_f1': val_f1,
            'optimizer_state_dict': optimizer.state_dict(), 
        }
        np.save(training_metadata_save_path, metadata)
        print(f"save best model in {model_weights_save_path}")
    else:
        patience_counter += 1
    if patience_counter >= patience:
        print(f"Early stopping triggered after {epoch+1} epochs")
        break

print("\nend of training.")

history_data = {
    'train_losses': train_losses,
    'val_losses': val_losses,
    'val_ious': val_ious,
    'val_precisions': val_precisions,
    'val_recalls': val_recalls,
    'val_f1s': val_f1s,
    'best_val_iou': best_val_iou
}
np.save(history_save_path, history_data)
print(f"train history store in '{history_save_path}'")

# ============================================================
# 8. Visualization of training history
# ============================================================
def plot_training_history(history_data):
    epochs = range(1, len(history_data['train_losses']) + 1)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Plot Training & Validation Loss
    axes[0, 0].plot(epochs, history_data['train_losses'], 'bo-', label='Training Loss')
    axes[0, 0].plot(epochs, history_data['val_losses'], 'ro-', label='Validation Loss')
    axes[0, 0].set_title('Model Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Plot Validation IoU
    axes[0, 1].plot(epochs, history_data['val_ious'], 'go-', label='Validation IoU')
    axes[0, 1].set_title('Validation IoU')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('IoU')
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # Plot Validation Precision
    axes[0, 2].plot(epochs, history_data['val_precisions'], 'mo-', label='Validation Precision')
    axes[0, 2].set_title('Validation Precision')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('Precision')
    axes[0, 2].legend()
    axes[0, 2].grid(True)

    # Plot Validation Recall
    axes[1, 0].plot(epochs, history_data['val_recalls'], 'co-', label='Validation Recall')
    axes[1, 0].set_title('Validation Recall')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Recall')
    axes[1, 0].legend()
    axes[1, 0].grid(True)

    # Plot Validation F1-Score
    axes[1, 1].plot(epochs, history_data['val_f1s'], 'yo-', label='Validation F1-Score')
    axes[1, 1].set_title('Validation F1-Score')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('F1-Score')
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    # Plot All Validation Metrics Together
    axes[1, 2].plot(epochs, history_data['val_ious'], 'g-', label='IoU')
    axes[1, 2].plot(epochs, history_data['val_precisions'], 'm--', label='Precision')
    axes[1, 2].plot(epochs, history_data['val_recalls'], 'c:', label='Recall')
    axes[1, 2].plot(epochs, history_data['val_f1s'], 'y-.', label='F1-Score')
    axes[1, 2].set_title('All Validation Metrics')
    axes[1, 2].set_xlabel('Epoch')
    axes[1, 2].set_ylabel('Score')
    axes[1, 2].legend()
    axes[1, 2].grid(True)

    plt.tight_layout()
    plt.show()
plot_training_history(history_data)

# ============================================================
# 9. Load best model safely and evaluate
# ============================================================
def load_model_safely(model_class, weights_path, metadata_path, in_channels=3, out_channels=1, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Loading model to device: {device}")
    model = model_class(in_channels=in_channels, out_channels=out_channels).to(device)
    model.eval() 
    print(f"Loading weights from: {weights_path}")
    try:
        checkpoint = torch.load(weights_path, map_location=device, weights_only=True)
        model.load_state_dict(checkpoint)
        print("Weights loaded successfully with weights_only=True.")
    except Exception as e:
        print(f"Error loading weights with weights_only=True: {e}")
        raise e
    print(f"Loading metadata from: {metadata_path}")
    try:
        metadata = np.load(metadata_path, allow_pickle=True).item()
        print("Metadata loaded successfully.")
    except Exception as e:
        print(f"Error loading metadata: {e}")
        print("No metadata available, returning empty dict.")
        metadata = {}

    return model, metadata

try:
    model, metadata = load_model_safely(
        model_class=UNet,
        weights_path=model_weights_save_path,
        metadata_path=training_metadata_save_path,
        device=device
    )
    print(f"Loaded model state from {model_weights_save_path}")
    print(f"Loaded metadata from {training_metadata_save_path}")
    print(f"Best model's metadata: {metadata}")
except Exception as e:
    print(f"Failed to load model safely: {e}")
    print("Attempting legacy load method...")
    try:
        with torch.serialization.safe_globals([np.dtype, np.core.multiarray.scalar]):
            checkpoint = torch.load(model_weights_save_path, map_location=device, weights_only=True)
        model.load_state_dict(checkpoint)
        model.eval()
        print("Legacy load successful (using combined file or different path?)")
    except:
        print("Legacy load also failed. Please check paths and file formats.")


final_val_loss, final_val_iou, final_val_precision, final_val_recall, final_val_f1 = validate_epoch(model, val_loader, criterion, device)
print(f"Loss: {final_val_loss:.6f}")
print(f"IoU: {final_val_iou:.4f}")
print(f"Precision: {final_val_precision:.4f}")
print(f"Recall: {final_val_recall:.4f}")
print(f"F1-Score: {final_val_f1:.4f}")

def visualize_predictions(model, dataset, num_samples=4, device='cuda'):
    model.eval()
    fig, axes = plt.subplots(3, num_samples, figsize=(15, 10))
    
    for i in range(num_samples):
        img, true_mask = dataset[i]
        img_tensor = img.unsqueeze(0).to(device) 
        
        with torch.no_grad():
            pred_mask = model(img_tensor)
        
        pred_mask = (pred_mask.cpu().squeeze().numpy() > 0.5).astype(np.uint8)
        true_mask = true_mask.squeeze().numpy().astype(np.uint8)
        img = img.permute(1, 2, 0).numpy()

        axes[0, i].imshow(img)
        axes[0, i].set_title("Original Image")
        axes[0, i].axis('off')

        axes[1, i].imshow(true_mask, cmap='gray')
        axes[1, i].set_title("Ground Truth Mask")
        axes[1, i].axis('off')

        axes[2, i].imshow(pred_mask, cmap='gray')
        axes[2, i].set_title("Predicted Mask")
        axes[2, i].axis('off')

    plt.tight_layout()
    plt.show()

visualize_predictions(model, val_dataset, num_samples=4, device=device)